# Pipeline Tag Prediction
## Tagging the Untagged Model Cards with the Fine-Tuned ModernBERT Model

**DATASCI 266: Natural Language Processing with Deep Learning**

UC Berkeley, School of Information

---

This notebook takes the cleaned untagged model cards and runs them through the fine-tuned ModernBERT model to predict a pipeline tag and confidence score for each one, the same idea as the baseline and RoBERTa inference notebooks, just swapping in ModernBERT.

Loads the saved model artifacts from `ModernBert_Model_Artifacts/`, no retraining:
- `config.json`
- `model.safetensors`
- `tokenizer.json`
- `tokenizer_config.json`

Steps:
1. Install/import dependencies
2. Load the fine-tuned ModernBERT model and tokenizer
3. Load the untagged dataset
4. Tokenize and predict tags with confidence scores
5. Assemble and save the final dataset
6. Quick sanity checks

## 0. Setup

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import joblib
import warnings

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 1. Load the Fine-Tuned ModernBERT Model and Tokenizer

Loading directly from the `ModernBert_Model_Artifacts` folder on Drive. `AutoModelForSequenceClassification` and `AutoTokenizer` both read straight from that directory since it has `config.json`, `model.safetensors`, `tokenizer.json`, and `tokenizer_config.json`.

`training_args.bin` isn't needed here, that's leftover from the `Trainer` checkpoint and has no bearing on inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'
MODERNBERT_DIR = f'{DRIVE_DIR}/modernbert_pipeline_tag_model'

tokenizer = AutoTokenizer.from_pretrained(MODERNBERT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODERNBERT_DIR)
model.to(device)
model.eval()

print('Loaded ModernBERT model and tokenizer from', MODERNBERT_DIR)
print(f'Number of labels: {model.config.num_labels}')

Mounted at /content/drive


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Loaded ModernBERT model and tokenizer from /content/drive/MyDrive/266-pipeline-tag-prediction/modernbert_pipeline_tag_model
Number of labels: 10


## 2. Resolve Label Mapping

Checking whether `config.json` already has real tag names in `id2label`, or just generic placeholders like `LABEL_0`. If it's generic, we fall back to `label_encoder.joblib` from the baseline artifacts, since both models were trained on the same label encoding.

In [ ]:
id2label = model.config.id2label
print('id2label from model config:')
print(id2label)

generic_labels = all(str(v).startswith('LABEL_') for v in id2label.values())

if generic_labels:
    print()
    print('Config only has generic labels, falling back to label_encoder.joblib')
    label_encoder = joblib.load(f'{DRIVE_DIR}/label_encoder.joblib')
    # label_encoder.classes_ is ordered by the same integer index used during training
    id2label = {i: cls for i, cls in enumerate(label_encoder.classes_)}
    print('Resolved id2label from label encoder:')
    print(id2label)
else:
    print()
    print('Config already has real tag names, using as is.')

id2label from model config:
{0: 'automatic-speech-recognition', 1: 'image-classification', 2: 'image-text-to-text', 3: 'robotics', 4: 'sentence-similarity', 5: 'text-classification', 6: 'text-generation', 7: 'text-to-image', 8: 'token-classification', 9: 'translation'}

Config already has real tag names, using as is.


## 3. Load the Untagged Dataset

Same cleaned untagged dataset used for the baseline predictions. Columns are `modelId`, `text`, `char_len`, `word_count`, model input is `text`.

In [ ]:
df_untagged = pd.read_parquet(f'{DRIVE_DIR}/model_cards_untagged_cleaned.parquet')

print(f'Shape: {df_untagged.shape}')
df_untagged.head()

Shape: (115620, 4)


,modelId,text,char_len,word_count
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,# DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF\n...,1820,182
1,namlevan888/blockassist-bc-lethal_durable_rave...,# Gensyn BlockAssist\n\nGensyn's BlockAssist i...,169,17
2,ElenaSenger/career-path-representation-mpnet-d...,# career-path-representation-mpnet-decorte\nTh...,842,67
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr...,# llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-cre...,1436,139
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,## About\n\n<!-- ### quantize_version: 2 -->\n...,5703,488


## 4. Tokenize and Predict

Running inference in batches to avoid blowing up GPU memory on the full untagged set. ModernBERT's `8,192`-token window means far fewer cards get truncated compared to RoBERTa. However, our fine-tuned ModernBERT model was trained using a context window with a maximum of 512 tokens due to the minimal impact found from text length on text classification during prior error analysis. For each card we take the softmax over the 10 classes, the predicted tag is the argmax, and the confidence is that max probability.

In [ ]:
BATCH_SIZE = 16
MAX_LENGTH = 512

texts = df_untagged['text'].tolist()

all_pred_idx = []
all_confidence = []

with torch.no_grad():
    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        ).to(device)

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

        batch_pred_idx = torch.argmax(probs, dim=-1).cpu().numpy()
        batch_confidence = torch.max(probs, dim=-1).values.cpu().numpy()

        all_pred_idx.extend(batch_pred_idx.tolist())
        all_confidence.extend(batch_confidence.tolist())

        if (i // BATCH_SIZE) % 20 == 0:
            print(f'Processed {i + len(batch_texts)} / {len(texts)}')

print('Inference complete.')

Processed 16 / 115620
Processed 336 / 115620
Processed 656 / 115620
Processed 976 / 115620
Processed 1296 / 115620
Processed 1616 / 115620
Processed 1936 / 115620
Processed 2256 / 115620
Processed 2576 / 115620
Processed 2896 / 115620
Processed 3216 / 115620
Processed 3536 / 115620
Processed 3856 / 115620
Processed 4176 / 115620
Processed 4496 / 115620
Processed 4816 / 115620
Processed 5136 / 115620
Processed 5456 / 115620
Processed 5776 / 115620
Processed 6096 / 115620
Processed 6416 / 115620
Processed 6736 / 115620
Processed 7056 / 115620
Processed 7376 / 115620
Processed 7696 / 115620
Processed 8016 / 115620
Processed 8336 / 115620
Processed 8656 / 115620
Processed 8976 / 115620
Processed 9296 / 115620
Processed 9616 / 115620
Processed 9936 / 115620
Processed 10256 / 115620
Processed 10576 / 115620
Processed 10896 / 115620
Processed 11216 / 115620
Processed 11536 / 115620
Processed 11856 / 115620
Processed 12176 / 115620
Processed 12496 / 115620
Processed 12816 / 115620
Processed 13

In [ ]:
df_untagged['predicted_tag'] = [id2label[idx] for idx in all_pred_idx]
df_untagged['confidence'] = all_confidence
df_untagged['confidence_pct'] = (df_untagged['confidence'] * 100).round(2)

print('Predictions complete.')
df_untagged[['modelId', 'predicted_tag', 'confidence_pct']].head(10)

Predictions complete.


,modelId,predicted_tag,confidence_pct
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,text-generation,98.03
1,namlevan888/blockassist-bc-lethal_durable_rave...,text-generation,86.00
2,ElenaSenger/career-path-representation-mpnet-d...,sentence-similarity,98.56
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr...,text-generation,99.36
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,text-generation,99.35
5,4everStudent/Qwen3-4B-GRPO-chess-puzzle,text-generation,99.94
6,javasop/orbital-cli,text-generation,51.04
7,little1d/C,automatic-speech-recognition,61.39
8,munish0838/Qwen-2.5-1.5B-cenv-trl-grpo-v3,text-generation,91.35
9,phanerozoic/threshold-parity6,text-classification,77.60


A quick look at the confidence distribution and predicted tag counts, useful to sanity check whether the model is confident overall or hedging a lot on this unseen population of cards.

In [ ]:
print(df_untagged['confidence_pct'].describe())
print()
print(df_untagged['predicted_tag'].value_counts())

count    115620.000000
mean         90.936266
std          15.127038
min          20.290000
25%          89.280000
50%          98.760000
75%          99.880000
max         100.000000
Name: confidence_pct, dtype: float64

predicted_tag
text-generation                 77114
image-text-to-text              10549
text-classification              6395
text-to-image                    5832
image-classification             4900
automatic-speech-recognition     3051
robotics                         2794
token-classification             2548
translation                      1261
sentence-similarity              1176
Name: count, dtype: int64


In [ ]:
# Spot check a handful of low-confidence predictions, these are the ones most worth a manual look later
df_untagged.sort_values('confidence_pct').head(5)[['modelId', 'predicted_tag', 'confidence_pct', 'text']]

,modelId,predicted_tag,confidence_pct,text
103033,structlearning/isonetpp-gmn_match-ptc_mr-large,text-generation,20.29,# ISONeT++ Model: gmn_match on ptc_mr\n\n T...
68144,Raphaelogbodo/GNN-IL-DES-models,text-classification,20.58,This contains trained models on ILs and transf...
108249,qykcboi/XSMoE,automatic-speech-recognition,20.70,# Efficient-Multimodal-Streaming-Recommendatio...
63452,MLSpeech/FALCON-weights,text-generation,20.83,# FALCON — pretrained checkpoints\n\nPretraine...
19683,structlearning/isonetpp-gmn_match-ptc_fr-large,text-classification,21.50,# ISONeT++ Model: gmn_match on ptc_fr\n\n T...


## 5. Assemble and Save the Final Dataset

Final columns: `modelId`, the card text, and the predicted tag with confidence. Keeping `predicted_tag` and `confidence_pct` separate rather than folding them into one string, so confidence stays usable as a number for filtering or thresholding later, same convention as the baseline output.

Saved as both a parquet and a CSV, matching the baseline inference notebook.

In [ ]:
final_df = df_untagged[['modelId', 'text', 'predicted_tag', 'confidence_pct']].copy()
final_df = final_df.rename(columns={'text': 'card_text_clean'})

print(f'Final dataset shape: {final_df.shape}')
final_df.head()

Final dataset shape: (115620, 4)


,modelId,card_text_clean,predicted_tag,confidence_pct
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,# DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF\n...,text-generation,98.03
1,namlevan888/blockassist-bc-lethal_durable_rave...,# Gensyn BlockAssist\n\nGensyn's BlockAssist i...,text-generation,86.00
2,ElenaSenger/career-path-representation-mpnet-d...,# career-path-representation-mpnet-decorte\nTh...,sentence-similarity,98.56
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr...,# llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-cre...,text-generation,99.36
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,## About\n\n<!-- ### quantize_version: 2 -->\n...,text-generation,99.35


In [ ]:
final_df.to_parquet(f'{DRIVE_DIR}/untagged_model_cards_predicted_modernbert.parquet', index=False)
final_df.to_csv(f'{DRIVE_DIR}/untagged_model_cards_predicted_modernbert.csv', index=False)

print(f'Saved to {DRIVE_DIR}/untagged_model_cards_predicted_modernbert.parquet')
print(f'Saved to {DRIVE_DIR}/untagged_model_cards_predicted_modernbert.csv')

Saved to /content/drive/MyDrive/266-pipeline-tag-prediction/untagged_model_cards_predicted_modernbert.parquet
Saved to /content/drive/MyDrive/266-pipeline-tag-prediction/untagged_model_cards_predicted_modernbert.csv


## 6. Quick Sanity Checks

Comparing ModernBERT's predicted tag distribution against the baseline's, if you've already run the baseline inference notebook and have `untagged_model_cards_predicted.parquet` on Drive. Big divergences between the two are worth a closer look, since they're predicting on the exact same untagged cards.

In [ ]:
import os

baseline_path = f'{DRIVE_DIR}/untagged_model_cards_predicted.parquet'

if os.path.exists(baseline_path):
    baseline_df = pd.read_parquet(baseline_path)
    merged = final_df.merge(
        baseline_df[['modelId', 'predicted_tag', 'confidence_pct']],
        on='modelId',
        suffixes=('_modernbert', '_baseline')
    )
    agreement = (merged['predicted_tag_modernbert'] == merged['predicted_tag_baseline']).mean()
    print(f'Agreement rate between ModernBERT and baseline predictions: {agreement:.2%}')

    disagreements = merged[merged['predicted_tag_modernbert'] != merged['predicted_tag_baseline']]
    print(f'Number of disagreements: {len(disagreements)}')
    disagreements[['modelId', 'predicted_tag_baseline', 'confidence_pct_baseline',
                    'predicted_tag_modernbert', 'confidence_pct_modernbert']].head(10)
else:
    print('Baseline predictions file not found on Drive, skipping comparison.')
    print('Run NH_Untagged_Inference.ipynb first if you want this comparison.')

Agreement rate between ModernBERT and baseline predictions: 80.16%
Number of disagreements: 22935


### Notes for later use

- These are ModernBERT-model predictions, not ground truth. Treat `confidence_pct` as a filter, not a guarantee, low-confidence rows are the ones most likely to be genuinely ambiguous or out of distribution relative to the training set.
- ModernBERT truncates at 8,192 tokens, so only unusually long model cards lose their tail end during prediction, this is a meaningfully smaller issue than with RoBERTa's 512-token limit.
- Since the model was trained only on the top 10 pipeline tags, every prediction here is forced into one of those 10 categories, even if a card actually belongs to a tag outside that set.
- If `ModernBert_Model_Artifacts/` is missing `config.json`, `model.safetensors`, `tokenizer.json`, or `tokenizer_config.json`, this notebook will fail at Section 1. Re-save the fine-tuned model with `model.save_pretrained()` and `tokenizer.save_pretrained()` if that happens.